In [7]:
### IMPORTS

import requests
from itext2kg import iText2KG_Star,DocumentsDistiller
import gc
import yaml
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, field_validator
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from difflib import SequenceMatcher
import os
import re
import asyncio
from typing import List, Dict, Any, Optional
from itext2kg import iText2KG_Star
from itext2kg.logging_config import setup_logging, get_logger
from langchain_core.caches import BaseCache
import fitz
from neo4j import GraphDatabase



# Suppress Pydantic schema warning
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()
ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()


In [ ]:
### Yt transcription's reader 

from youtube_transcript_api import YouTubeTranscriptApi

video_id = "ZRu2aGzBV34"

transcript_list = YouTubeTranscriptApi.get_transcript(video_id)

transcript = " ".join([item['text'] for item in transcript_list])

print(transcript)

with open("AgileVideos/Mastering AgilePDF's Concepts for PMP® and PMI-ACP®.txt", "w", encoding="utf-8") as f:
    f.write(transcript)

In [11]:
### PDF READER

def read_pdf_to_text(pdf_path: str):
    try:
        text = ""
        doc = fitz.open(pdf_path)
        
        total_pages = len(doc)  
        
        for page_num in range(total_pages):
            page = doc.load_page(page_num)
            text += page.get_text()
            text += "\n"  
            
        doc.close()
        
        with open("text.txt", "w", encoding="utf-8") as f:
            f.write(text)
            
        print(f"PDF content extracted and saved to text.txt")
        print(f"Extracted {len(text)} characters from {total_pages} pages")
        
    except Exception as e:
        print(f"Error reading PDF: {e}")

In [12]:
### LOAD CONFIG

def load_config(config_path="config.yml"):
    try:
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        print("config.yml not found. Creating sample config for Ollama...")

        sample_config = {
            'llm': {
                'model_type': 'ollama',
                'model_name': 'llama3.1',
                'format': 'json',
                'temperature': 0.1,
                'max_tokens': 1000,
                'timeout': 45,
                'max_retries': 3
            },
            'paths': {
                'output_dir': './olderFiles'
            }
        }

        with open(config_path, "w") as f:
            yaml.dump(sample_config, f, default_flow_style=False)

        print("Sample config.yml created. Update it if needed.")
        return sample_config


In [13]:
### LLM AND EMBEDINGS

def initialize_llm(config):
    if not config:
        return None, None

    llm_config = config.get('llm', {})

    try:
        if llm_config.get('model_type') == "ollama":
            llm = ChatOllama(
                model=llm_config.get('model_name', 'llama3.1'),
                temperature=llm_config.get('temperature', 0.1)
            )

            embeddings = OllamaEmbeddings(model=llm_config.get('model_name', 'llama3.1'))

            print("LLM and Embeddings initialized successfully (Ollama)")
            return llm, embeddings
        else:
            print("Only 'ollama' model type is supported in this setup")
            return None, None
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None, None


In [14]:
### CUSTOM SCHEMA AND PROMPT
class Scrum(BaseModel):
    title: Optional[str] = Field(default="")
    abstract: Optional[str] = Field(default="")
    scrum_roles: Optional[str] = Field(default="")
    scrum_artifacts: Optional[str] = Field(default="")
    scrum_events: Optional[str] = Field(default="")
    scrum_values: Optional[str] = Field(default="")
    methodology: Optional[str] = Field(default="")
    processes: Optional[str] = Field(default="")
    responsibilities: Optional[str] = Field(default="")
    workflow: Optional[str] = Field(default="")
    introduction: Optional[str] = Field(default="")
    background: Optional[str] = Field(default="")
    objectives: Optional[str] = Field(default="")
    requirements: Optional[str] = Field(default="")
    implementation: Optional[str] = Field(default="")
    results: Optional[str] = Field(default="")
    conclusions: Optional[str] = Field(default="")
    key_concepts: Optional[str] = Field(default="")
    definitions: Optional[str] = Field(default="")
    examples: Optional[str] = Field(default="")

    @field_validator('*', mode='before')
    @classmethod
    def convert_none_to_empty_string(cls, v):
        if v is None:
            return ""
        if isinstance(v, (list, dict)) and not v:
            return ""
        return v

def get_semantic_distiller_query():
    return '''
# DIRECTIVES : 
- Act like an experienced Scrum expert and information extractor
- You are analyzing a document about Scrum software development process
- Extract information into structured categories that will help build a knowledge graph
- Focus on roles, responsibilities, artifacts, events, and their relationships
- If you do not find specific information for a category, return empty string ""
- Be precise and use Scrum terminology when possible
- CRITICAL: Always return valid strings for all fields, never return null, None, lists, or objects
- If a field has no relevant content, use an empty string ""
- Avoid redundancy - if information is already captured in one field, don't repeat in others
'''

In [15]:
### HELPER FUNCTIONS
def safe_get_name(entity, default="unknown"):
    try:
        if entity is None:
            return default
        if hasattr(entity, "name") and entity.name:
            return str(entity.name)
        if isinstance(entity, str):
            return entity
        if isinstance(entity, dict) and "name" in entity:
            return str(entity["name"])
    except:
        pass
    return default

TRIVIAL_OBJECTS = {"yes", "no", "none", "all_the_time", "sometimes_present", "unknown"}

def clean_entity(entity: str) -> str:
    if not entity:
        return ""
    entity = re.sub(r'\s+', ' ', entity.strip())
    entity = re.sub(r'^(The|A|An)\s+', '', entity, flags=re.IGNORECASE)
    entity = re.sub(r'[,;:.!?]+$', '', entity)
    entity = re.sub(r'\b(team|group|department)\b', lambda m: m.group().title(), entity, flags=re.IGNORECASE)
    entity = entity.title()
    if entity.lower() in TRIVIAL_OBJECTS:
        return ""
    return entity

def enhanced_clean_predicate(predicate: str) -> str:
    if not predicate:
        return "relates_to"

    predicate = predicate.lower().strip()
    predicate = re.sub(r'[^\w\s]', '', predicate)
    predicate = re.sub(r'\s+', '_', predicate)

    grammar_fixes = {
        'from_the_perspective_of': 'relates_to',
        'has_product_owner': 'includes',
        'is_practic': 'practices',
        'complet': 'completes',
        'manag': 'manages',
        'involve': 'involves',
        'emphasizes_iterative_and_incremental_progress_towards_a_well_defined_goal': 'emphasizes',
        'prioritizing_and_refining_the_product_backlog': 'manages_backlog',
        'facilitating_the_proces': 'facilitates',
        'responsible_for_prioritiz': 'prioritizes',
        'responsible_for_estimating_story_size': 'estimates_size',
        'assigns_value_to_each_story': 'assigns_value',
        'prioritize_stories_in_scrum': 'prioritizes_stories',
        'consider_story_value_and_size': 'considers_value_and_size',
        'prioritize_tasks_based_on_business_value_and_complexity': 'prioritizes_by_value',
        'estimate_size_using_story_points_or_hour': 'estimates_using_points',
        'assign_value_based_on_business_value_or_impact': 'assigns_business_value'
    }

    if predicate in grammar_fixes:
        return grammar_fixes[predicate]

    mappings = {
        'is_a': 'is', 'has_a': 'has', 'contain': 'contains', 'belong': 'belongs_to',
        'work': 'works_on', 'manage': 'manages', 'create': 'creates', 'use': 'uses',
        'participate': 'participates_in', 'attend': 'attends', 'build': 'builds',
        'define': 'defines', 'prioritize': 'prioritizes', 'facilitate': 'facilitates'
    }

    for k, v in mappings.items():
        if k in predicate:
            return v

    if predicate.endswith('ing') and len(predicate) > 5:
        predicate = predicate[:-3]
    elif predicate.endswith('ed') and len(predicate) > 5:
        predicate = predicate[:-2]
    elif predicate.endswith('s') and len(predicate) > 3 and not predicate.endswith('ss'):
        predicate = predicate[:-1]

    return predicate

def validate_triple_semantically(subject: str, predicate: str, obj: str) -> bool:
    invalid_patterns = [
        (r'Product Owner', r'is_scrum_master.*', r'.*'),
        (r'Scrum Master', r'is_product_owner.*', r'.*'),
        (r'.*', r'from_the_perspective_of', r'.*'),
        (r'.*', r'has_product_owner', r'Development Team'),  
        (r'Product Owner', r'.*', r'Product Owner'),
        (r'Scrum Master', r'.*', r'Scrum Master'),
        (r'Sprint Review', r'follow', r'Sprint Planning'), 
        (r'Sprint Planning', r'preceded_by', r'Daily Scrum')  
    ]

    for s_pattern, p_pattern, o_pattern in invalid_patterns:
        if (re.match(s_pattern, subject, re.IGNORECASE) and 
            re.match(p_pattern, predicate, re.IGNORECASE) and 
            re.match(o_pattern, obj, re.IGNORECASE)):
            return False

    return True

def is_valid_triple(subject: str, predicate: str, obj: str) -> bool:
    if len(subject) < 2 or len(obj) < 2: 
        return False
    if subject.lower() == obj.lower(): 
        return False

    bad_words = {'it', 'this', 'that', 'they', 'he', 'she', 'something', 'anything',
                 'everything', 'nothing', 'somewhere', 'anywhere', 'everyone', 'anyone'}
    if subject.lower() in bad_words or obj.lower() in bad_words: 
        return False

    if predicate in ['is', 'has', 'does'] and len(predicate) < 4: 
        return False

    if not validate_triple_semantically(subject, predicate, obj):
        return False

    return True

def deduplicate_semantic_blocks(blocks: List[str], similarity_threshold: float = 0.75) -> List[str]:
    if not blocks:
        return blocks

    unique_blocks = []
    for block in blocks:
        is_duplicate = False
        block_clean = block.lower().strip()

        for existing_block in unique_blocks:
            existing_clean = existing_block.lower().strip()
            similarity = SequenceMatcher(None, block_clean, existing_clean).ratio()

            if similarity > similarity_threshold:
                is_duplicate = True
                if len(block) > len(existing_block):
                    unique_blocks.remove(existing_block)
                    unique_blocks.append(block)
                break

        if not is_duplicate:
            unique_blocks.append(block)

    return unique_blocks

def save_triples(triples, filename="extracted_triples.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Knowledge Graph Triples\n# Total: {len(triples)} triples\n")
        f.write("# Format: (Subject) --[Predicate]--> (Object)\n\n")
        for i, (s, p, o) in enumerate(triples, 1):
            f.write(f"{i:3d}. ({s}) --[{p}]--> ({o})\n")
        f.write(f"\n# Extraction completed. {len(triples)} unique triples saved.\n")


class PipelineError(Exception):
    pass

async def robust_distillation(chunk: str, distiller, max_retries: int = 3):
    for attempt in range(max_retries):
        try:
            cleaned_chunk = chunk.replace("{", "[").replace("}", "]")

            distilled_article = await distiller.distill(
                documents=[cleaned_chunk],
                IE_query=get_semantic_distiller_query(),
                output_data_structure=Scrum
            )

            return distilled_article

        except asyncio.TimeoutError as e:
            print(f"    → Attempt {attempt + 1} timed out: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"    → Retrying in {wait_time} seconds...")
                await asyncio.sleep(wait_time)
        except Exception as e:
            error_type = type(e).__name__
            error_msg = str(e)[:100] + "..." if len(str(e)) > 100 else str(e)
            print(f"    → Attempt {attempt + 1} failed ({error_type}): {error_msg}")

            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"    → Retrying in {wait_time} seconds...")
                await asyncio.sleep(wait_time)
            else:
                print(f"    → All {max_retries} attempts failed, using fallback")
                raise PipelineError(f"Distillation failed after {max_retries} attempts: {error_type} - {error_msg}")

def process_chunks_in_batches(chunks, batch_size=5):
    for i in range(0, len(chunks), batch_size):
        yield chunks[i:i + batch_size]



def filter_semantic_blocks(blocks: List[str], min_length: int = 30) -> List[str]:
    filtered_blocks = []
    for block in blocks:
        if not block or len(block.strip()) < min_length:
            continue

        trivial_patterns = [
            r'^\s*$',  
            r'^\w+:\s*(none|unknown|yes|no|sometimes|all_the_time)\s*$',  
            r'^[^:]+:\s*$'  
        ]
        if any(re.search(pat, block, flags=re.IGNORECASE) for pat in trivial_patterns):
            continue

        filtered_blocks.append(block)
    return filtered_blocks

def create_semantic_blocks(distilled_article: Scrum) -> List[str]:
    semantic_blocks = []
    article_dict = distilled_article.model_dump()

    block_config = {
        'title': 'Document Title',
        'abstract': 'Executive Summary', 
        'scrum_roles': 'Scrum Roles and Responsibilities',
        'scrum_artifacts': 'Scrum Artifacts and Deliverables',
        'scrum_events': 'Scrum Events and Ceremonies',
        'scrum_values': 'Scrum Values and Principles',
        'methodology': 'Methodology and Framework',
        'processes': 'Processes and Workflows',
        'responsibilities': 'Roles and Responsibilities',
        'workflow': 'Workflow and Procedures',
        'introduction': 'Introduction',
        'background': 'Background',
        'objectives': 'Objectives and Goals',
        'requirements': 'Requirements and Constraints',
        'implementation': 'Implementation Details',
        'results': 'Results',
        'conclusions': 'Conclusions',
        'key_concepts': 'Key Concepts and Definitions',
        'definitions': 'Terminology and Definitions',
        'examples': 'Examples and Use Cases'
    }

    for key, value in article_dict.items():
        if value and isinstance(value, str) and len(value.strip()) > 20:
            cleaned_value = value.replace("{", "[").replace("}", "]").strip()

            prefix = block_config.get(key, key.replace('_', ' ').title())
            semantic_block = f"{prefix}: {cleaned_value}"

            semantic_blocks.append(semantic_block)

    return semantic_blocks

class ETACalculator:
    def __init__(self, total_items: int):
        self.total_items = total_items
        self.start_time = time.time()
        self.processing_times = []

    def update(self, current_item: int, item_processing_time: float = None):
        if item_processing_time:
            self.processing_times.append(item_processing_time)

        elapsed = time.time() - self.start_time
        avg_time = elapsed / current_item if current_item > 0 else 0
        remaining = avg_time * (self.total_items - current_item)

        return {
            'elapsed_seconds': elapsed,
            'elapsed_minutes': elapsed / 60,
            'avg_time_per_item': avg_time,
            'remaining_seconds': remaining,
            'remaining_minutes': remaining / 60,
            'progress_percent': (current_item / self.total_items) * 100
        }

    def get_eta_string(self, current_item: int) -> str:
        eta = self.update(current_item)
        return f"Progress: {eta['progress_percent']:.1f}% | Elapsed: {eta['elapsed_minutes']:.1f}min | ETA: {eta['remaining_minutes']:.1f}min"



In [39]:
### MAIN
async def main(input_source: str = "text.txt"):
    print("Starting Enhanced KG Extraction with Semantic Blocks Pipeline (Fixed)\n")

    if os.path.exists(input_source):
        with open(input_source, "r", encoding="utf-8") as f:
            text = f.read().strip()
    else:
        text = input_source.strip()

    if len(text) < 20:
        print("Text too short")
        return []

    config = load_config()  
    llm, embeddings = initialize_llm(config) 
    print(f"LLM initialized: {config['llm']['model_name']}")

    all_triples = []
    start_time = time.time()

    print(f"\n=== STEP 1: Document Distillation (Memory Optimized) ===")

    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk + sentence) <= 1000:  
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    print(f"Text chunked: {len(text)} chars in {len(chunks)} chunks")

    eta_distiller = ETACalculator(len(chunks))

    try:
        distiller = DocumentsDistiller(llm_model=llm)
    except ImportError as e:
        print(f"Failed to import DocumentsDistiller: {e}")
        return []

    all_semantic_blocks = []
    successful_chunks = 0
    BATCH_SIZE = 5 

    for batch_num, chunk_batch in enumerate(process_chunks_in_batches(chunks, BATCH_SIZE)):
        print(f"\n{'='*60}")
        print(f"STARTING DISTILLATION BATCH {batch_num + 1}/{(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE}")
        print(f"Processing {len(chunk_batch)} chunks in this batch")
        print(f"{'='*60}")

        batch_blocks = []

        for i, chunk in enumerate(chunk_batch):
            chunk_start_time = time.time()
            global_chunk_idx = batch_num * BATCH_SIZE + i
            print(f"Distilling chunk {global_chunk_idx + 1}/{len(chunks)}...")

            try:
                distilled_article = await robust_distillation(chunk, distiller)

                if distilled_article:
                    chunk_blocks = create_semantic_blocks(distilled_article)
                    batch_blocks.extend(chunk_blocks)
                    successful_chunks += 1
                    print(f"Generated {len(chunk_blocks)} semantic blocks")
                else:
                    print("No semantic blocks generated; creating fallback")
                    if len(chunk.strip()) > 50:
                        fallback_block = f"Content Block: {chunk.strip()[:500]}..."
                        batch_blocks.append(fallback_block)

            except PipelineError as e:
                print(f"Using fallback block: {e}")
                if len(chunk.strip()) > 50:
                    fallback_block = f"Content Block: {chunk.strip()[:500]}..."
                    batch_blocks.append(fallback_block)

            eta_info = eta_distiller.get_eta_string(global_chunk_idx + 1)
            print(f"{eta_info}")

        all_semantic_blocks.extend(batch_blocks)
        print(f"\nDISTILLATION BATCH {batch_num + 1} COMPLETED!")
        print(f"Added {len(batch_blocks)} blocks to collection")
        print(f"Total blocks so far: {len(all_semantic_blocks)}")
        print(f"{'='*60}")

        del batch_blocks
        gc.collect()

    print(f"\nDistillation Summary:")
    print(f"- Total semantic blocks created: {len(all_semantic_blocks)}")
    print(f"- Successful chunks: {successful_chunks}/{len(chunks)}")

    print(f"Filtering and deduplicating {len(all_semantic_blocks)} semantic blocks...")
    all_semantic_blocks = filter_semantic_blocks(all_semantic_blocks, min_length=30)
    all_semantic_blocks = deduplicate_semantic_blocks(all_semantic_blocks, similarity_threshold=0.75)
    print(f"Remaining semantic blocks after filtering and deduplication: {len(all_semantic_blocks)}")

    print("\n=== STEP 2: STAR Knowledge Graph Extraction (Enhanced Error Handling) ===")

    if not all_semantic_blocks:
        print("No semantic blocks available, creating basic blocks from chunks")
        all_semantic_blocks = [f"Text Chunk: {chunk[:300]}..." for chunk in chunks if len(chunk.strip()) > 50]

    try:
        logger = get_logger(__name__)
        star_model = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)

        print(f"Processing {len(all_semantic_blocks)} semantic blocks with STAR...")

        batch_size = 10  
        batches = [all_semantic_blocks[i:i+batch_size] for i in range(0, len(all_semantic_blocks), batch_size)]

        eta_star = ETACalculator(len(batches))
        total_star_triples = 0

        for batch_idx, batch in enumerate(batches):
            batch_start_time = time.time()
            print(f"\n{'='*60}")
            print(f"PROCESSING STAR BATCH {batch_idx + 1}/{len(batches)}")
            print(f"Processing {len(batch)} semantic blocks in this batch")
            print(f"{'='*60}")

            batch_triples = 0

            for block_idx, block in enumerate(batch):
                print(f"Processing block {block_idx + 1}/{len(batch)} (Global: {batch_idx * batch_size + block_idx + 1}/{len(all_semantic_blocks)})")

                try:
                    kg_result = await star_model.build_graph(
                        sections=[block],  
                        ent_threshold=0.5,
                        rel_threshold=0.4,
                        observation_date="2025-01-15",
                    )

                    if hasattr(kg_result, 'relationships') and kg_result.relationships:
                        block_triples = 0
                        for relationship in kg_result.relationships:
                            try:
                                s = clean_entity(safe_get_name(getattr(relationship, "startEntity", None)))
                                o = clean_entity(safe_get_name(getattr(relationship, "endEntity", None)))
                                p = enhanced_clean_predicate(safe_get_name(getattr(relationship, "name", None), "relates_to"))

                                if is_valid_triple(s, p, o):
                                    all_triples.append((s, p, o))
                                    batch_triples += 1
                                    block_triples += 1

                            except AttributeError as attr_error:
                                print(f"Relationship attribute error: {attr_error}")
                                continue
                            except Exception as rel_error:
                                print(f"Relationship processing error: {type(rel_error).__name__}: {rel_error}")
                                continue

                        print(f"Extracted {block_triples} triples from this block")
                    else:
                        print(f"Block {block_idx + 1} produced no relationships")

                except asyncio.TimeoutError as timeout_error:
                    print(f"Block {block_idx + 1} timed out: {timeout_error}")
                    continue
                except ImportError as import_error:
                    print(f"Import error for block {block_idx + 1}: {import_error}")
                    continue
                except Exception as block_error:
                    error_type = type(block_error).__name__
                    error_msg = str(block_error)[:100] + "..." if len(str(block_error)) > 100 else str(block_error)
                    print(f"Block {block_idx + 1} failed ({error_type}): {error_msg}")
                    continue

            total_star_triples += batch_triples

            batch_time = time.time() - batch_start_time
            eta_info = eta_star.get_eta_string(batch_idx + 1)

            print(f"\nSTAR BATCH {batch_idx + 1} COMPLETED!")
            print(f"Extracted {batch_triples} valid triples from this batch")
            print(f"{eta_info}")
            print(f"Total triples so far: {total_star_triples}")
            print(f"{'='*60}")

            del batch
            gc.collect()

        print(f"\nSTAR Extraction Summary:")
        print(f"- Total triples extracted: {total_star_triples}")
        print(f"- Successful batches: {len(batches)}")

    except ImportError as import_error:
        print(f"STAR import failed: {import_error}")
        print("Cannot proceed without STAR - please check your itext2kg installation")
        return []
    except Exception as star_error:
        error_type = type(star_error).__name__
        error_msg = str(star_error)[:200] + "..." if len(str(star_error)) > 200 else str(star_error)
        print(f"STAR initialization failed ({error_type}): {error_msg}")
        print("Cannot proceed without STAR - please check your setup")
        return []

    print(f"\n=== STEP 3: Post-Processing ===")

    seen, unique_triples = set(), []
    skipped_invalid = 0

    for triple in all_triples:
        s, p, o = triple

        s_clean = clean_entity(s)
        p_clean = enhanced_clean_predicate(p)
        o_clean = clean_entity(o)

        clean_triple = (s_clean, p_clean, o_clean)

        if not is_valid_triple(s_clean, p_clean, o_clean):
            skipped_invalid += 1
            continue

        if clean_triple not in seen:
            seen.add(clean_triple)
            unique_triples.append(clean_triple)

    final_triples = unique_triples

    elapsed_time = time.time() - start_time
    print(f"\nProcessing completed in {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
    print(f"Total triples extracted: {len(all_triples)}")
    print(f"Invalid triples skipped: {skipped_invalid}")
    print(f"Final unique valid triples: {len(final_triples)}")

    save_triples(final_triples, "Triples and Semantic Blocks/blogs_triples_v1.txt")

    if final_triples:
        print(f"\nSample extracted triples:")
        for i, (s, p, o) in enumerate(final_triples[:10], 1):
            print(f"   {i:2d}. ({s}) --[{p}]--> ({o})")
        if len(final_triples) > 10:
            print(f"   ... and {len(final_triples) - 10} more")

    if all_semantic_blocks:
        with open("Triples and Semantic Blocks/blogs_semanticBlocks_v1.txt", "w", encoding="utf-8") as f:
            f.write(f"# Semantic Blocks Generated\n# Total: {len(all_semantic_blocks)} blocks\n\n")
            for i, block in enumerate(all_semantic_blocks, 1):
                f.write(f"Block {i}:\n{block}\n{'-'*50}\n\n")
        print(f"\nSemantic blocks saved to: blogs_semanticBlocks_v1.txt")

    return final_triples

In [40]:
if __name__ == "__main__":
    import nest_asyncio

    nest_asyncio.apply()
    # 
    # pdf_file = "AgilePDF's/EBM Guide 2020_1.pdf"  
    # read_pdf_to_text(pdf_file)

    async def run_star_pipeline():
        triples = await main("text.txt")
        print(f"\nPipeline completed! Found {len(triples)} triples")
        return triples


    loop = asyncio.get_event_loop()
    triples = loop.run_until_complete(run_star_pipeline())

Starting Enhanced KG Extraction with Semantic Blocks Pipeline (Fixed)

LLM and Embeddings initialized successfully (Ollama)
LLM initialized: llama3.1

=== STEP 1: Document Distillation (Memory Optimized) ===
Text chunked: 21779 chars in 25 chunks

STARTING DISTILLATION BATCH 1/5
Processing 5 chunks in this batch
Distilling chunk 1/25...
Generated 0 semantic blocks
Progress: 4.0% | Elapsed: 1.9min | ETA: 46.4min
Distilling chunk 2/25...
    → Attempt 1 failed (AttributeError): 'NoneType' object has no attribute 'items'
    → Retrying in 1 seconds...
Generated 7 semantic blocks
Progress: 8.0% | Elapsed: 4.4min | ETA: 50.3min
Distilling chunk 3/25...
    → Attempt 1 failed (AttributeError): 'NoneType' object has no attribute 'items'
    → Retrying in 1 seconds...
    → Attempt 2 failed (AttributeError): 'NoneType' object has no attribute 'items'
    → Retrying in 2 seconds...
    → Attempt 3 failed (AttributeError): 'NoneType' object has no attribute 'items'
    → All 3 attempts failed, u

In [6]:
class Neo4jClient:
    def __init__(self, uri="bolt://localhost:7687", user="neo4j", password="12345678"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def create_triples(self, triples, label_mapping=None, rel_mapping=None):
        if label_mapping is None:
            label_mapping = {}
        if rel_mapping is None:
            rel_mapping = {}

        auto_labels = self._auto_detect_labels(triples)
        for entity, label in auto_labels.items():
            if entity not in label_mapping:
                label_mapping[entity] = label

        auto_rels = self._auto_detect_relationships(triples)
        for pred, rel_type in auto_rels.items():
            if pred not in rel_mapping:
                rel_mapping[pred] = rel_type

        print(f" Processing {len(triples)} triples with {len(set(label_mapping.values()))} entity types")

        with self.driver.session() as session:
            for subj, pred, obj in triples:
                session.execute_write(lambda tx: self._create_single_triple(tx, subj, pred, obj, label_mapping, rel_mapping))

    def _auto_detect_labels(self, triples):
        label_mapping = {}

        entities = set()
        for subj, _, obj in triples:
            entities.add(subj)
            entities.add(obj)

        patterns = {
            "Person": [r'.*schwaber.*', r'.*sutherland.*', r'^ken$', r'^jeff$'],
            "Role": [r'.*master.*', r'.*owner.*', r'.*developer.*', r'.*team.*', r'.*member.*', r'.*stakeholder.*'],
            "Artifact": [r'.*backlog.*', r'.*increment.*', r'.*goal.*', r'.*definition.*'],
            "Event": [r'.*sprint.*', r'.*meeting.*', r'.*planning.*', r'.*review.*', r'.*retrospective.*', r'daily scrum'],
            "Framework": [r'^scrum$', r'agile.*', r'.*methodology.*', r'.*framework.*'],
            "Value": [r'commitment$', r'focus$', r'openness$', r'respect$', r'courage$', r'transparency$'],
            "Organization": [r'.*company.*', r'.*organization.*'],
            "Document": [r'.*guide.*', r'.*documentation.*']
        }

        for entity in entities:
            entity_lower = entity.lower()
            matched = False

            for label, entity_patterns in patterns.items():
                if any(re.match(pattern, entity_lower, re.IGNORECASE) for pattern in entity_patterns):
                    label_mapping[entity] = label
                    matched = True
                    break

            if not matched:
                label_mapping[entity] = "Entity"

        return label_mapping

    def _auto_detect_relationships(self, triples):
        rel_mapping = {}
        predicates = set(pred for _, pred, _ in triples)

        for pred in predicates:
            cleaned = self._clean_predicate_name(pred)
            rel_mapping[pred] = cleaned

        return rel_mapping

    def _clean_predicate_name(self, predicate):
        common_mappings = {
            'is_part_of': 'PART_OF',
            'works_on': 'WORKS_ON', 
            'has_role': 'HAS_ROLE',
            'related_to': 'RELATED_TO',
            'responsible_for': 'RESPONSIBLE_FOR',
            'co_founded': 'CO_FOUNDED',
            'has_scrum_master': 'HAS_SCRUM_MASTER',
            'include': 'INCLUDES',
            'serve': 'SERVES'
        }

        if predicate in common_mappings:
            return common_mappings[predicate]

        cleaned = predicate.lower()

        abbreviations = {
            'responsibility': 'RESP',
            'responsibilities': 'RESP',
            'definition': 'DEF', 
            'development': 'DEV',
            'management': 'MGMT',
            'organization': 'ORG',
            'implementation': 'IMPL',
            'communication': 'COMM',
            'collaboration': 'COLLAB',
            'effectiveness': 'EFF',
            'accountability': 'ACCT',
            'understanding': 'UNDER'
        }

        for long_word, short_word in abbreviations.items():
            cleaned = cleaned.replace(long_word, short_word)

        cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', cleaned)
        cleaned = re.sub(r'_+', '_', cleaned)  
        cleaned = cleaned.strip('_').upper()

        if not cleaned or cleaned == '_':
            cleaned = "RELATES_TO"

        if len(cleaned) > 50:
            parts = cleaned.split('_')
            if len(parts) > 3:
                cleaned = f"{parts[0]}_{parts[len(parts)//2]}_{parts[-1]}"

            if len(cleaned) > 50:
                cleaned = cleaned[:47] + "REL"

        return cleaned

    def _create_single_triple(self, tx, subj, pred, obj, label_mapping, rel_mapping):
        subj_label = label_mapping.get(subj, "Entity")
        obj_label = label_mapping.get(obj, "Entity") 
        rel_type = rel_mapping.get(pred, "RELATES_TO")

        query = f"""
        MERGE (s:{subj_label} {{name: $subj}})
        MERGE (o:{obj_label} {{name: $obj}})
        MERGE (s)-[r:{rel_type}]->(o)
        RETURN s, r, o
        """
        tx.run(query, subj=subj, obj=obj)



label_mapping = {
    "Scrum Master": "Role",
    "Product Owner": "Role",
    "Development Team": "Team", 
    "Company": "Organization",
    "Sprint": "Event",
    "Nexus Guide": "Document",
    "Integrated Increment": "Artifact",
    "Scrum Team": "Team",
    "Product Backlog": "Artifact",
    "Nexus": "Framework",
}

rel_mapping = {
    "published_in": "PUBLISHED_IN",
    "works_on": "WORKS_ON",
    "manages": "MANAGES",
    "is_part_of": "PART_OF", 
    "related_to": "RELATED_TO",
    "deliver": "DELIVERS",
    "contributes_to": "CONTRIBUTES_TO",
}


triples = []
with open("Triples and Semantic Blocks/documentEBM_triples_v1.txt", "r", encoding="utf-8") as f:
    for line in f:
        try:
            if line.strip().startswith(tuple('0123456789')):
                line = line.split('. ', 1)[1] if '. ' in line else line

            subj, rest = line.split("--[")
            pred, obj = rest.split("]-->")
            triples.append((subj.strip()[1:-1], pred.strip(), obj.strip()[1:-1]))
        except:
            continue

print(f"Loaded {len(triples)} triples")


neo4j_client = Neo4jClient(uri="bolt://localhost:7687", user="neo4j", password="12345678")
neo4j_client.create_triples(triples, label_mapping, rel_mapping)
neo4j_client.close()

print("Done! Try these queries in Neo4j Browser:")
print("MATCH (n) RETURN labels(n), count(*) ORDER BY count(*) DESC")
print("MATCH (r:Role) RETURN r.name")
print("MATCH (n {name: 'Product Owner'})-[rel]-(connected) RETURN n, rel, connected LIMIT 10")


Loaded 199 triples
 Processing 199 triples with 9 entity types
Done! Try these queries in Neo4j Browser:
MATCH (n) RETURN labels(n), count(*) ORDER BY count(*) DESC
MATCH (r:Role) RETURN r.name
MATCH (n {name: 'Product Owner'})-[rel]-(connected) RETURN n, rel, connected LIMIT 10
